<a href="https://colab.research.google.com/github/ASHIMTOM7/AIandML/blob/main/Copy_of_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anvil-uplink

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [ ]:
import numpy as np
import anvil
import cv2
from tqdm import tqdm
import os
import imutils
import matplotlib.pyplot as plt
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from keras.models import load_model


In [ ]:

def crop_img(img):
	"""
	Finds the extreme points on the image and crops the rectangular out of them
	"""
	gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
	gray = cv2.GaussianBlur(gray, (3, 3), 0)

	# threshold the image, then perform a series of erosions +
	# dilations to remove any small regions of noise
	thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
	thresh = cv2.erode(thresh, None, iterations=2)
	thresh = cv2.dilate(thresh, None, iterations=2)

	# find contours in thresholded image, then grab the largest one
	cnts=cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
	cnts=imutils.grab_contours(cnts)
	c = max(cnts, key=cv2.contourArea)

	# find the extreme points
	extLeft = tuple(c[c[:, :, 0].argmin()][0])
	extRight = tuple(c[c[:, :, 0].argmax()][0])
	extTop = tuple(c[c[:, :, 1].argmin()][0])
	extBot = tuple(c[c[:, :, 1].argmax()][0])
	ADD_PIXELS = 0
	new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()

	return new_img

In [ ]:

#p=input()
#image = (cv2.imread(p))
#image=crop_img(image)
#image = cv2.bilateralFilter(image, 2, 50, 50)
#image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
#image = cv2.resize(image, (200, 200))
#image =np.array(image)/255.0
lab=['glioma', 'meningioma', 'notumor', 'pituitary']

In [ ]:
#p=model.predict(np.array([image]))
#print(p)

In [ ]:
#y_classes = p.argmax(axis=-1)
#print(y_classes)

In [ ]:
#print(lab[y_classes[0]])

In [ ]:
from google.colab import drive
import anvil.media
import anvil.server
from PIL import Image
anvil.server.connect("BNLTQEAP73DLV6QNL2MAGIYL-JFDT3PMLPAMHN3LH")


@anvil.server.callable
def classify_image(file):

  with anvil.media.TempFile(file) as filename:
    model = load_model('/content/drive/MyDrive/model-08-0.98-0.08.h5')
    image = Image.open(filename)
    image =np.array(image)
    image = crop_img(image)
    image = cv2.bilateralFilter(image, 2, 50, 50)
    image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
    image = cv2.resize(image, (200, 200))
    image = np.array(image)/255.0
    p=model.predict(np.array([image]))
    y_classes = p.argmax(axis=-1)
    return(lab[y_classes[0]])